In [4]:
import os
print("CWD:", os.getcwd())

CWD: /Users/user/data/embeddings


In [8]:
import torch
import pandas as pd
from pathlib import Path

# You are in /Users/user/data/embeddings
emb_path = Path("all_embeddings_COMPLETE.pt")
embeddings = torch.load(emb_path, map_location="cpu")

print("Embeddings:", embeddings.shape, embeddings.dtype)

# 1) Save a copy with a clear name (optional)
out_pt = Path("embeddings_all.pt")
torch.save(embeddings, out_pt)

# 2) Minimal metadata CSV (only embedding index)
out_csv = Path("embeddings_all_index.csv")
df = pd.DataFrame({"embedding_idx": range(embeddings.shape[0])})
df.to_csv(out_csv, index=False)

print("Saved:", out_pt.resolve())
print("Saved:", out_csv.resolve())

Embeddings: torch.Size([2390, 1280]) torch.float32
Saved: /Users/user/data/embeddings/embeddings_all.pt
Saved: /Users/user/data/embeddings/embeddings_all_index.csv


In [11]:
from Bio import SeqIO
import pandas as pd
from pathlib import Path

# FASTA paths relative to /Users/user/data/embeddings
lassa_fasta = Path("../processed/lassa_cleaned.fasta")
ebola_fasta = Path("../processed/ebola_cleaned.fasta")

def load_fasta(filepath, virus_name):
    seqs = []
    for record in SeqIO.parse(str(filepath), "fasta"):
        seq_str = str(record.seq).strip()
        if seq_str:
            seqs.append({
                "id": record.id,
                "sequence": seq_str,
                "length": len(seq_str),
                "virus": virus_name,
            })
    return seqs

lassa_seqs = load_fasta(lassa_fasta, "Lassa")
ebola_seqs = load_fasta(ebola_fasta, "Ebola")

all_seqs = lassa_seqs + ebola_seqs
print("Lassa:", len(lassa_seqs), "Ebola:", len(ebola_seqs), "All:", len(all_seqs))

# Save metadata with embedding_idx that matches row order
df = pd.DataFrame(all_seqs)
df["embedding_idx"] = range(len(df))

out_csv = Path("all_metadata.csv")
df.to_csv(out_csv, index=False)

print("Saved:", out_csv.resolve(), "rows:", len(df))
print(df.head(3))

Lassa: 780 Ebola: 1610 All: 2390
Saved: /Users/user/data/embeddings/all_metadata.csv rows: 2390
                     id                                           sequence  \
0    X52400|s|LIII|3417  APGILGIDCAFNLLFGKCRNQDGTDCDILPRSSSCYGSDEYCPYCT...   
1  OL774861|s|LIII|3371  AQWILGYWIALCTNQTFGVTTFKTHGSDNHHDNECFQGSEVILVDS...   
2  MK107965|s|LIII|3407  IYISRRLRILEALFGDQTIRMGQIVTFFQEVPHVIEEVMNIVLIAL...   

   length  virus  embedding_idx  
0    1064  Lassa              0  
1    1059  Lassa              1  
2    1097  Lassa              2  


In [12]:
import torch
import pandas as pd
from pathlib import Path

# Inputs (already in this folder)
emb_pt = Path("all_embeddings_COMPLETE.pt")
meta_csv = Path("all_metadata.csv")

# Load
embeddings = torch.load(emb_pt, map_location="cpu")          # (2390, 1280)
meta = pd.read_csv(meta_csv)                                # 2390 rows

# Filter Lassa
lassa_meta = meta[meta["virus"] == "Lassa"].copy().reset_index(drop=True)
lassa_indices = lassa_meta["embedding_idx"].to_list()

print("All embeddings:", embeddings.shape)
print("Lassa rows:", len(lassa_indices))

# Slice embeddings for Lassa (keeps original order)
lassa_embeddings = embeddings[lassa_indices].contiguous()
print("Lassa embeddings:", lassa_embeddings.shape)

# Save Lassa-only outputs
out_pt = Path("lassa_embeddings.pt")
out_csv = Path("lassa_metadata.csv")

torch.save(lassa_embeddings, out_pt)

# re-index embedding_idx to 0..779 for Lassa-only file
lassa_meta["embedding_idx"] = range(len(lassa_meta))
lassa_meta.to_csv(out_csv, index=False)

print("Saved:", out_pt.resolve())
print("Saved:", out_csv.resolve())

All embeddings: torch.Size([2390, 1280])
Lassa rows: 780
Lassa embeddings: torch.Size([780, 1280])
Saved: /Users/user/data/embeddings/lassa_embeddings.pt
Saved: /Users/user/data/embeddings/lassa_metadata.csv
